# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from itertools import chain

# Load & Inspect data

In [0]:
df_raw = spark.table("workspace.bronze.crm_sales_details_raw")
df_raw.display()

In [0]:
df_raw.printSchema()

In [0]:
print(f'Table contains {df_raw.count()} rows')
print(f'Table contains {df_raw.select(["sls_ord_num", "sls_prd_key"]).distinct().count()} distinct ids')

# Silver Transformation

## Rename columns

In [0]:
column_map = {
    'sls_ord_num': 'order_id',
    'sls_prd_key': 'product_key',
    'sls_cust_id': 'customer_id',
    'sls_order_dt': 'order_date',
    'sls_ship_dt': 'ship_date',
    'sls_due_dt': 'due_date',
    'sls_sales': 'sales_amount',
    'sls_quantity': 'quantity',
    'sls_price': 'price'
}

df = df_raw.select([F.col(c).alias(column_map.get(c, c)) for c in df_raw.columns])

df.schema.fields

## Trim strings

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

df.display()

## Cast date columns

In [0]:
date_cols = [c for c in df.columns if "_date" in c]   # OR c.endswith("_date")

for c in date_cols:
    df = df.withColumn(
        c,
        F.when(
            (F.col(c).isNull()) | (F.col(c) == 0) | (F.length(F.col(c).cast("string")) != 8),
            F.lit(None).cast("date")
        ).otherwise(
            F.to_date(F.col(c).cast("string"), "yyyyMMdd")
        )
    )

df.limit(10).display()
df.schema.fields

## Clean "price"

In [0]:
df.filter(F.col("sales_amount") / F.col("quantity") != F.col("price")).display()

In [0]:
df = (
    df
    .withColumn("price", F.abs(F.col("price").cast("double")))
    .withColumn("sales_amount", F.abs(F.col("sales_amount").cast("double")))
    .withColumn("sales_amount", F.when(F.col("sales_amount").isNull(), F.col("price") * F.col("quantity")).otherwise(F.col("sales_amount")))
    .withColumn("quantity", F.abs(F.col("quantity").cast("integer")))
    .withColumn("quantity", F.when(F.col("sales_amount") == F.col("price"), F.lit(1)).otherwise(F.col("quantity")))
    .withColumn("quantity", F.when((F.col("price") == 9) & (F.col("sales_amount") == 18), F.lit(2)).otherwise(F.col("quantity")))
)

df.select("price", "quantity", "sales_amount").filter((F.col("sales_amount") / F.col("quantity") != F.col("price"))).display()

In [0]:
df.display()

## Null check

In [0]:
df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).display()

In [0]:
df.count()

# Write table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

# Sanity check

In [0]:
%sql
SELECT * FROM workspace.silver.crm_sales